# Construire une solution à la main sur `1_example`
On raisonne comme le concours : **qui peut remplir quel besoin ? qu'est-ce que ça coûte ? quels choix s'excluent ?** Puis on écrit la solution et on la fait vérifier.

**Prérequis** : `datasets/1_example.json` et `test_solution.py` (à la racine du projet), `pandas`.

## 1. Charger l'instance

In [15]:
from pathlib import Path
from itertools import combinations
import sys, json
import pandas as pd
pd.set_option("display.width", 150)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from test_solution import getSolutionScore

DATASET = ROOT_DIR / "datasets" / "1_example.json"
instance = json.loads(DATASET.read_text())
energy_cap = instance["energyCap"]
datasets = pd.DataFrame(instance["datasets"]).rename(columns={"id": "dataset_id", "dataType": "data_type", "isCopyrighted": "is_copyrighted"})
models = pd.DataFrame(instance["models"])[["id", "value", "energyCost"]].rename(columns={"id": "model_id", "energyCost": "energy_cost"})
requirements = pd.DataFrame([(m["id"], t, b["lowerBound"], b["upperBound"]) for m in instance["models"]
                             for t, b in m["dataRequirements"].items()], columns=["model_id", "data_type", "lower", "upper"])
print("plafond d'énergie :", energy_cap)
display(models); display(datasets); display(requirements)

plafond d'énergie : 184958


,model_id,value,energy_cost
0,0,6770,70153
1,1,16940,88492
2,2,9615,48961
3,3,3416,68214
4,4,4476,25558


,dataset_id,data_type,size,is_copyrighted
0,0,n,1524,1
1,1,t,867,0
2,2,t,1149,1
3,3,i,740,1
4,4,i,918,0


,model_id,data_type,lower,upper
0,0,n,1462,1749
1,1,t,1888,2330
2,1,n,1892,2003
3,2,t,1973,2500
4,3,n,1520,1810
5,4,i,670,995


## 2. Qui peut remplir quel besoin ?
Pour chaque besoin `(modèle, type)`, on cherche les **combinaisons de datasets du même type** dont la somme des tailles tombe dans `[lower, upper]`. Il n'y a que 5 datasets : on peut tout essayer.

In [16]:
rows = []
for r in requirements.itertuples():
    same_type = datasets[datasets.data_type == r.data_type]
    valid = []
    for k in range(1, len(same_type) + 1):
        for combo in combinations(same_type.itertuples(), k):
            total = sum(d.size for d in combo)
            if r.lower <= total <= r.upper:
                valid.append(dict(datasets=[d.dataset_id for d in combo], somme=total,
                                  libre=all(d.is_copyrighted == 0 for d in combo)))
    rows.append(dict(modèle=r.model_id, type=r.data_type, lower=r.lower, upper=r.upper,
                     datasets_du_type=dict(zip(same_type.dataset_id, same_type["size"])), combinaisons_valides=valid))
feasible = pd.DataFrame(rows)
feasible

,modèle,type,lower,upper,datasets_du_type,combinaisons_valides
0,0,n,1462,1749,{0: 1524},"[{'datasets': [0], 'somme': 1524, 'libre': Fal..."
1,1,t,1888,2330,"{1: 867, 2: 1149}","[{'datasets': [1, 2], 'somme': 2016, 'libre': ..."
2,1,n,1892,2003,{0: 1524},[]
3,2,t,1973,2500,"{1: 867, 2: 1149}","[{'datasets': [1, 2], 'somme': 2016, 'libre': ..."
4,3,n,1520,1810,{0: 1524},"[{'datasets': [0], 'somme': 1524, 'libre': Fal..."
5,4,i,670,995,"{3: 740, 4: 918}","[{'datasets': [3], 'somme': 740, 'libre': Fals..."


**Lecture**
- Modèle 0 (`n`) et modèle 3 (`n`) : seul le dataset 0 convient, et il est sous droits d'auteur. Les deux modèles se **disputent le même dataset**.
- Modèle 1 : son besoin `t` se remplit avec les datasets 1 + 2 (somme 2016), mais son besoin `n` demande entre 1892 et 2003 alors que le seul dataset `n` fait 1524. **Aucune combinaison de datasets ne suffit.** Seule une source de type `n` peut le couvrir.
- Modèle 2 (`t`) veut aussi les datasets 1 + 2 : même conflit avec le modèle 1.
- Modèle 4 (`i`) : le dataset 4 (918, libre) ou le dataset 3 (740, sous droits) suffisent, mais pas les deux ensemble (1658 > 995). Choisir le dataset 4 garde la valeur complète.

## 3. Ce que rapporte et ce que coûte chaque modèle
Valeur obtenue selon le cas : données libres (valeur entière) ou dataset sous droits / données générées (valeur ÷ 2, arrondie à l'inférieur).

In [17]:
m = models.set_index("model_id").copy()
m["valeur_si_libre"] = m.value
m["valeur_si_malus"] = m.value // 2
m["valeur_par_énergie_malus"] = (m.valeur_si_malus / m.energy_cost * 1e4).round(1)    # ×10⁴ pour lire les nombres
m["part_du_plafond"] = (m.energy_cost / energy_cap).round(2)
m

,value,energy_cost,valeur_si_libre,valeur_si_malus,valeur_par_énergie_malus,part_du_plafond
model_id,,,,,,
0,6770,70153,6770,3385,482.5,0.38
1,16940,88492,16940,8470,957.1,0.48
2,9615,48961,9615,4807,981.8,0.26
3,3416,68214,3416,1708,250.4,0.37
4,4476,25558,4476,2238,875.7,0.14


## 4. Les plans possibles
Les conflits de la partie 2 réduisent beaucoup les choix. Voici trois plans, écrits à la main. Chaque plan est un dictionnaire au format du concours :

```
dataMappings  : [dataset, modèle]   ce dataset est donné à ce modèle
modelMappings : [source, cible]     ce modèle sert de source de données à la cible
```

In [18]:
# Plan A : on entraîne 0, 2 et 4 directement, avec les datasets
plan_a = {"dataMappings": [[0, 0],          # dataset 0 (n) -> modèle 0
                           [1, 2], [2, 2],  # datasets 1 + 2 (t) -> modèle 2
                           [4, 4]],         # dataset 4 (i, libre) -> modèle 4
          "modelMappings": []}

# Plan B : le modèle 1 (le plus précieux) avec une source pour son besoin n
plan_b = {"dataMappings": [[1, 1], [2, 1],  # datasets 1 + 2 (t) -> modèle 1
                           [0, 0],          # dataset 0 (n) -> modèle 0, qui devient la source
                           [4, 4]],         # dataset 4 (i, libre) -> modèle 4
          "modelMappings": [[0, 1]]}        # modèle 0 sert de source n au modèle 1

# Plan C : comme A mais avec le modèle 3 à la place du 0 (même dataset 0, valeur plus faible)
plan_c = {"dataMappings": [[0, 3], [1, 2], [2, 2], [4, 4]], "modelMappings": []}

for name, plan in [("A", plan_a), ("B", plan_b), ("C", plan_c)]:
    score, valid, message = getSolutionScore(json.dumps(plan), DATASET.read_text(), show_details=False)
    print(f"Plan {name} : score = {score:6d} | valide = {valid} {message}")

Plan A : score =  12668 | valide = True 
Plan B : score =  12946 | valide = True 
Plan C : score =  10991 | valide = True 


## 5. Pourquoi le plan B gagne
- Plan A : modèles 0, 2, 4 → 3385 + 4807 + 4476 = **12 668**. Le modèle 2 est dévalué (dataset 2 sous droits), le 0 aussi (dataset 0).
- Plan B : modèle 1 + sa source 0 + modèle 4 → 8470 + 4476 = **12 946**. Le modèle 1 vaut 16 940, divisé par deux car il utilise une source et un dataset sous droits (les deux malus ne se cumulent pas). La valeur du modèle 0, utilisé comme source, n'est pas comptée, mais son énergie l'est.
- Le plan B dépense presque tout le plafond : on peut le voir dans le détail ci-dessous.

In [19]:
score, valid, message = getSolutionScore(json.dumps(plan_b), DATASET.read_text(), show_details=True)
print("score =", score, "| valide =", valid)


Total value: 12946
Energy used: 184203 / 184958 (99.59%)
Number of used models: 3 / 5 (60.00%)
  Number of devalued models: 2 (66.67%)
  Number of models used as data source: 1 (33.33%)
Number of used datasets: 4 / 5 (80.00%)
score = 12946 | valide = True


## 6. Écrire le fichier de solution

In [20]:
out_dir = ROOT_DIR / "solutions"; out_dir.mkdir(exist_ok=True)
out = out_dir / "1_example_main.json"
out.write_text(json.dumps(plan_b))
print("écrit :", out)

écrit : /home/thgoncal/euro-info-code-contest/solutions/1_example_main.json


## À retenir
1. **Les types découpent le problème** : les quatre types de données sont quatre réserves indépendantes.
2. **Les conflits comptent plus que les scores** : un même dataset ne sert qu'un modèle, donc choisir un modèle en exclut d'autres.
3. **Une source remplace des datasets manquants** mais coûte son énergie et divise par deux la valeur de la cible.
4. **Le meilleur plan n'est pas le plus évident** : ici, un modèle à gros potentiel (1), pénalisé et appuyé sur une source, l'emporte.